In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# Locate project directory
PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

RAW_DIR = PROJECT_DIR / "data" / "raw"
CLEAN_DIR = PROJECT_DIR / "data" / "cleaned"

CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# Load datasets
suppliers = pd.read_csv(RAW_DIR / "suppliers.csv")
components = pd.read_csv(RAW_DIR / "components.csv")
purchase_orders = pd.read_csv(RAW_DIR / "purchase_orders.csv")
shipments = pd.read_csv(RAW_DIR / "shipments.csv")
inventory = pd.read_csv(RAW_DIR / "inventory_movements.csv")

datasets = {
    "Suppliers": suppliers,
    "Components": components,
    "Purchase Orders": purchase_orders,
    "Shipments": shipments,
    "Inventory Movements": inventory
}

print("All datasets loaded successfully.\n")

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

All datasets loaded successfully.

Suppliers: (40, 5)
Components: (500, 8)
Purchase Orders: (10000, 8)
Shipments: (10000, 7)
Inventory Movements: (19231, 7)


In [5]:
for name, df in datasets.items():
    print(f"\n{name} - Missing Values:")
    print(df.isnull().sum())


Suppliers - Missing Values:
SupplierID           0
SupplierName         0
Location             0
SupplierCategory     0
ReliabilityRating    0
dtype: int64

Components - Missing Values:
ComponentID      0
ComponentName    0
Category         0
UnitCost         0
CurrentStock     0
SafetyStock      0
LeadTimeDays     0
SupplierID       0
dtype: int64

Purchase Orders - Missing Values:
OrderID                 0
SupplierID              0
ComponentID             0
OrderDate               0
QuantityOrdered         0
UnitPrice               0
TotalCost               0
ExpectedDeliveryDate    0
dtype: int64

Shipments - Missing Values:
ShipmentID            0
OrderID               0
ActualDeliveryDate    0
QuantityReceived      0
ShippingCost          0
DefectiveUnits        0
DeliveryStatus        0
dtype: int64

Inventory Movements - Missing Values:
MovementID               0
ComponentID              0
MovementDate             0
MovementType             0
Quantity                 0
OrderID 

In [6]:
for name, df in datasets.items():
    print(f"{name}: {df.duplicated().sum()} duplicate rows")

Suppliers: 0 duplicate rows
Components: 0 duplicate rows
Purchase Orders: 0 duplicate rows
Shipments: 0 duplicate rows
Inventory Movements: 0 duplicate rows


In [11]:
for name, df in datasets.items():
    print(f"\n{name} - Data Types:")
    print(df.dtypes)


Suppliers - Data Types:
SupplierID               str
SupplierName             str
Location                 str
SupplierCategory         str
ReliabilityRating    float64
dtype: object

Components - Data Types:
ComponentID          str
ComponentName        str
Category             str
UnitCost         float64
CurrentStock       int64
SafetyStock        int64
LeadTimeDays       int64
SupplierID           str
dtype: object

Purchase Orders - Data Types:
OrderID                            str
SupplierID                         str
ComponentID                        str
OrderDate               datetime64[us]
QuantityOrdered                  int64
UnitPrice                      float64
TotalCost                      float64
ExpectedDeliveryDate    datetime64[us]
dtype: object

Shipments - Data Types:
ShipmentID                       str
OrderID                          str
ActualDeliveryDate    datetime64[us]
QuantityReceived               int64
ShippingCost                 float64
Defective

In [12]:

# Sort inventory movements chronologically
inventory_sorted = inventory.sort_values(
    ["ComponentID", "MovementDate", "MovementID"]
)

# Get the final recorded stock balance for each component
ending_stock = (
    inventory_sorted.groupby("ComponentID")
    ["StockAfterMovement"]
    .last()
    .reset_index()
)

ending_stock.rename(
    columns={"StockAfterMovement": "CalculatedStock"},
    inplace=True
)

print(ending_stock.head())
print("\nComponents with calculated stock:", len(ending_stock))


  ComponentID  CalculatedStock
0     CMP0001             1055
1     CMP0002             1715
2     CMP0003             1870
3     CMP0004             2211
4     CMP0005             1385

Components with calculated stock: 500


C:\Users\tanma\AppData\Local\Temp\ipykernel_20488\1210625894.py:14: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  ending_stock.rename(


In [13]:
stock_comparison = components[
    ["ComponentID", "CurrentStock"]
].merge(
    ending_stock,
    on="ComponentID",
    how="left",
    validate="one_to_one"
)

stock_comparison["StockDifference"] = (
    stock_comparison["CalculatedStock"]
    - stock_comparison["CurrentStock"]
)

print(stock_comparison.head())

print(
    "\nComponents with mismatched stock:",
    (stock_comparison["StockDifference"] != 0).sum()
)

  ComponentID  CurrentStock  CalculatedStock  StockDifference
0     CMP0001           407             1055              648
1     CMP0002           386             1715             1329
2     CMP0003           251             1870             1619
3     CMP0004            83             2211             2128
4     CMP0005           181             1385             1204

Components with mismatched stock: 500


In [14]:
# Preserve the original stock snapshot for auditing
components["OriginalCurrentStock"] = components["CurrentStock"]

# Update using the reconciled inventory balances
components = components.drop(columns=["CurrentStock"])

components = components.merge(
    ending_stock.rename(
        columns={"CalculatedStock": "CurrentStock"}
    ),
    on="ComponentID",
    how="left",
    validate="one_to_one"
)

print(components[
    ["ComponentID", "OriginalCurrentStock", "CurrentStock"]
].head())

  ComponentID  OriginalCurrentStock  CurrentStock
0     CMP0001                   407          1055
1     CMP0002                   386          1715
2     CMP0003                   251          1870
3     CMP0004                    83          2211
4     CMP0005                   181          1385


In [15]:
missing_stock = components["CurrentStock"].isna().sum()

negative_stock = (
    components["CurrentStock"] < 0
).sum()

stock_check = components[
    ["ComponentID", "CurrentStock"]
].merge(
    ending_stock,
    on="ComponentID",
    how="left"
)

stock_mismatches = (
    stock_check["CurrentStock"]
    != stock_check["CalculatedStock"]
).sum()

print("Missing stock values:", missing_stock)
print("Negative stock values:", negative_stock)
print("Stock mismatches:", stock_mismatches)

Missing stock values: 0
Negative stock values: 0
Stock mismatches: 0


In [16]:
components_clean = components.drop(
    columns=["OriginalCurrentStock"]
).copy()

print(components_clean.head())
print("\nShape:", components_clean.shape)

  ComponentID                  ComponentName        Category  UnitCost  \
0     CMP0001   Power Systems Component 0001   Power Systems   2586.72   
1     CMP0002      Propulsion Component 0002      Propulsion   1355.45   
2     CMP0003  Communications Component 0003  Communications   1380.52   
3     CMP0004      Propulsion Component 0004      Propulsion   4738.48   
4     CMP0005      Propulsion Component 0005      Propulsion   6487.68   

   SafetyStock  LeadTimeDays SupplierID  CurrentStock  
0           53            18     SUP012          1055  
1           21            34     SUP020          1715  
2           92            43     SUP025          1870  
3           76            20     SUP020          2211  
4           69            44     SUP020          1385  

Shape: (500, 8)


In [17]:

# Create cleaned dataset dictionary
cleaned_datasets = {
    "suppliers": suppliers,
    "components": components_clean,
    "purchase_orders": purchase_orders,
    "shipments": shipments,
    "inventory_movements": inventory
}

# Save cleaned datasets
for name, df in cleaned_datasets.items():

    output_path = CLEAN_DIR / f"{name}.csv"

    df.to_csv(
        output_path,
        index=False,
        date_format="%Y-%m-%d"
    )

    print(
        f"{name}.csv saved successfully! "
        f"Rows: {len(df)}"
    )

print("\nAll cleaned datasets exported!")


suppliers.csv saved successfully! Rows: 40
components.csv saved successfully! Rows: 500
purchase_orders.csv saved successfully! Rows: 10000
shipments.csv saved successfully! Rows: 10000
inventory_movements.csv saved successfully! Rows: 19231

All cleaned datasets exported!


In [18]:
for name, original_df in cleaned_datasets.items():

    file_path = CLEAN_DIR / f"{name}.csv"

    reloaded_df = pd.read_csv(file_path)

    print(
        f"{name}: "
        f"{len(reloaded_df)} rows, "
        f"{len(reloaded_df.columns)} columns"
    )

    assert len(reloaded_df) == len(original_df), (
        f"Row count mismatch in {name}"
    )

    assert list(reloaded_df.columns) == list(original_df.columns), (
        f"Column mismatch in {name}"
    )

print("\nAll exported datasets verified!")

suppliers: 40 rows, 5 columns
components: 500 rows, 8 columns
purchase_orders: 10000 rows, 8 columns
shipments: 10000 rows, 7 columns
inventory_movements: 19231 rows, 7 columns

All exported datasets verified!
